# numpy 강제 재설치 (버전 불일치 수정)
!pip install numpy==2.3.0 --force-reinstall -q
print('완료 - 런타임을 재시작하세요 (런타임 > 런타임 재시작)')

In [ ]:
# numpy 버전 충돌 수정 (런타임 재시작 필요)
!pip install numpy==1.26.4 -q
print('numpy 재설치 완료 - 런타임을 재시작하세요 (런타임 > 런타임 재시작)')

In [ ]:
# 저장소 클론 및 의존성 설치
!git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1.git
%cd /content/Hunyuan3D-2.1/hy3dshape
!pip install -r requirements.txt -q

In [ ]:
# 저장소 클론 및 의존성 설치
!git clone https://github.com/Tencent/Hunyuan3D-2.git
%cd Hunyuan3D-2
!pip install -r requirements.txt -q
!pip install -e . -q

In [ ]:
# custom rasterizer 빌드
!pip install ninja -q
%cd hy3dgen/texgen/custom_rasterizer
!python setup.py install
%cd /content/Hunyuan3D-2

In [ ]:
# 테스트 이미지 업로드
from google.colab import files
uploaded = files.upload()

import os
image_path = list(uploaded.keys())[0]
print(f'업로드된 이미지: {image_path}')

In [ ]:
# Hunyuan3D-2.1 추론
import sys, os

# hy3dshape 패키지 경로 설정
shape_path = '/content/Hunyuan3D-2.1/hy3dshape'
if shape_path not in sys.path:
    sys.path.insert(0, shape_path)

os.makedirs('/content/output', exist_ok=True)

from hy3dshape.pipelines import Hunyuan3DDiTFlowMatchingPipeline
from PIL import Image

# from_pretrained에 HuggingFace repo ID 직접 전달 (자동 다운로드)
print('모델 로드 중 (처음 실행 시 다운로드 포함, 수 분 소요)...')
pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained('tencent/Hunyuan3D-2.1')
pipeline = pipeline.to('cuda')

image = Image.open(bg_removed_path)
print('3D 생성 중...')
mesh = pipeline(image=image)[0]
mesh.export('/content/output/result.glb')
print('완료: /content/output/result.glb')

In [ ]:
# Hunyuan3D-2.1 추론 실행
import os
os.makedirs('/content/output', exist_ok=True)

from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline

pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    'tencent/Hunyuan3D-2',
    subfolder='hunyuan3d-dit-v2-1',
    use_safetensors=True,
)
pipeline = pipeline.to('cuda')

from PIL import Image
image = Image.open(bg_removed_path)

mesh = pipeline(image=image, num_inference_steps=30)[0]
mesh.export('/content/output/result.glb')
print('Shape 생성 완료: /content/output/result.glb')

In [ ]:
# (선택) 텍스처 입히기 - VRAM 여유 있을 때
from hy3dgen.texgen import Hunyuan3DPaintPipeline

import torch
torch.cuda.empty_cache()

paint_pipeline = Hunyuan3DPaintPipeline.from_pretrained('tencent/Hunyuan3D-2')
paint_pipeline = paint_pipeline.to('cuda')

import trimesh
mesh = trimesh.load('/content/output/result.glb')
textured_mesh = paint_pipeline(mesh=mesh, image=image)
textured_mesh.export('/content/output/result_textured.glb')
print('텍스처 적용 완료: /content/output/result_textured.glb')

In [ ]:
# 결과 확인 및 다운로드
import glob
output_files = glob.glob('/content/output/*.*')
print('생성된 파일:')
for f in output_files:
    size_mb = os.path.getsize(f) / 1024 / 1024
    print(f'  {f}  ({size_mb:.1f} MB)')

from google.colab import files
for f in output_files:
    files.download(f)